# Conditional LSTM Autoencoder — SrVO₃ Self-Energy Prediction

Trains a sequence-to-sequence LSTM autoencoder to predict the imaginary part of the
Matsubara self-energy Im Σ(iω) from the non-interacting Green's function Im G₀(iω),
conditioned on the number of CT-HYB warmup steps (`nwarmup`) as a convergence proxy.

**Pipeline:**
1. Load preprocessed SVO DMFT dataset (`data/dataSVO.npz`)  
2. Split: every 4th unique nwarmup value → train; remainder → test  
3. Build encoder–decoder LSTM with label concatenation  
4. Train 40 epochs, MSE loss  
5. Evaluate relative RMS residual per nwarmup group  
6. Save model to `models/trained_model_svo.keras`  

**Material:** SrVO₃ (d¹ correlated metal, β = 40 eV⁻¹)  
**Frequency grid:** 64 positive Matsubara points in (0, 10) eV

## 0. Configuration

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
DATA_PATH    = 'data/dataSVO.npz'
MODEL_OUT    = 'models/trained_model_svo.keras'
LATENT_DIM   = 64
NUM_EPOCHS   = 40
BATCH_SIZE   = 128
TRAIN_STRIDE = 4   # keep every 4th unique nwarmup value for training (1/4 split)
SEED         = 100


## 1. Load Data and Define Train / Test Split

We hold out 3 out of every 4 unique nwarmup values for testing.
This mimics a realistic scenario: the model must generalise to
convergence stages it has never seen during training.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt



##### read and preprocess for training and test
train_loaded = np.load('data/dataSVO.npz')
nwarmup_all = train_loaded['nwarupsteps']
real_sigma_all = train_loaded['realpart_sigma']
img_sigma_all = train_loaded['imgpart_sigma']




unique_nwarmups = np.unique(nwarmup_all)
nwarmups_train = unique_nwarmups[::4]


mask_train = np.isin(nwarmup_all, nwarmups_train)
mask_test = ~np.isin(nwarmup_all, nwarmups_train)

nwarmups_test = np.unique(nwarmup_all[mask_test])

print('train', np.unique(nwarmups_train))
print('test',np.unique(nwarmups_test))

## 2. Imports — TensorFlow / Keras

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Lambda, Embedding, BatchNormalization,LSTM, RepeatVector, TimeDistributed, concatenate, Reshape
from tensorflow.keras.losses import mse, binary_crossentropy
from tensorflow.keras import backend as K
import random

## 3. Build the Conditional LSTM Autoencoder

**Architecture (50,241 parameters):**
- Input 1: Im G₀(iω) sequence — shape `(64, 1)`
- Input 2: scalar nwarmup label (min-max normalised to [0,1])
- Label is broadcast via `RepeatVector` and concatenated with the input at every timestep
- Encoder LSTM (64 units) → hidden state `[h, c]`
- Decoder LSTM (64 units) initialised with encoder states
- `TimeDistributed(Dense(1))` → predicted Im Σ(iω) sequence

In [ ]:

# Define the model
latent_dim = 64

sequence_length = img_sigma_all.shape[1]
print(sequence_length)



# Set seeds for reproducibility
seed_value = 100
np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)




# Encoder


# Encoder
encoder_inputs = Input(shape=(sequence_length, 1))
label_inputs = Input(shape=(1,))  # Label input for the embedding
label_inputs_repeated = RepeatVector(sequence_length)(label_inputs)




# Concatenate the time series input with the embedded categorical input
merged_inputs = concatenate([encoder_inputs,label_inputs_repeated],axis=-1)

encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(merged_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = RepeatVector(sequence_length)(encoder_outputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True)
decoder_outputs = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = TimeDistributed(Dense(1))
decoder_outputs = decoder_dense(decoder_outputs)

# Define the full model
model = Model(inputs=[encoder_inputs, label_inputs], outputs=decoder_outputs)

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Summary of the model
model.summary()
for layer in model.layers:
    print(layer.name)

## 4. Prepare Training Arrays

Min-max normalise the nwarmup labels using the training-set range,
then reshape G₀ and Σ arrays to `(samples, 64, 1)` for the LSTM.

In [ ]:
#### train data
from sklearn.preprocessing import StandardScaler,MinMaxScaler, PowerTransformer
import joblib
'''
print(np.max(nwarmup_all,axis=0), np.min(nwarmup_all,axis=0))
scaler_x = PowerTransformer(method='box-cox')
scaler_y = PowerTransformer(method='box-cox')

'''
min_max_normal = lambda x: (x-min(x)/(max(x) - min(x)))
print(max(nwarmup_all), min(nwarmup_all))
all_labels = min_max_normal(nwarmup_all) #nwarmup

train_labels = all_labels[mask_train]
test_labels = all_labels[mask_test]

### preprocess

img_g0_all = train_loaded['imgpart_g0']  # load G0 imaginary part
X_all = img_g0_all #min_max_normal(img_train)
Y_all = img_sigma_all

X_train = X_all[mask_train]
X_test = X_all[mask_test]
X_train_scaled = X_train.reshape(-1,sequence_length , 1)
X_test_scaled = X_test.reshape(-1,sequence_length , 1)


Y_train = Y_all[mask_train]
Y_test = Y_all[mask_test]
Y_train_scaled = Y_train.reshape(-1,sequence_length , 1)
Y_test_scaled = Y_test.reshape(-1,sequence_length , 1)



## 5. Train

In [ ]:
# Train VAE
num_epochs = 40
batch_size = 128

history = model.fit([X_train_scaled,train_labels],Y_train_scaled,epochs=num_epochs, batch_size=batch_size,validation_split=0.2)


## 6. Plot Training and Validation Loss

In [ ]:
#### plot loss
plt.plot(history.history['loss'],'b-',label='training loss')
plt.plot(history.history['val_loss'],'r-',label='validation loss')
plt.legend()
plt.show()

## 7. Predict on Test Set and Evaluate

For each held-out nwarmup group we compute:
- Mean predicted profile (averaged over 6 spin-orbital channels per file)
- Per-channel standard deviation (error bars)
- Relative RMS residual: `rms((ŷ − y) / y)`

In [ ]:
y_pred =model.predict([X_test_scaled,test_labels])

In [ ]:
print(y_pred.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica'],
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 10,
    'axes.linewidth': 2.0,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'ytick.minor.width': 0.6,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'axes.labelpad': 10
})







y_test = Y_test_scaled

nwarmup_labels = nwarmup_all[mask_test]
unique_labels = np.unique(nwarmup_labels)

### frequency
freq=np.linspace(-161,161,2050)
mask = (freq > 0) & (freq < 10)
freq_sel = freq[mask]
rms = lambda x: np.sqrt(np.mean(x*x))

rms_mean =[]
y_truth_all=[]
y_pred_all=[]
y_pred_std_all=[]

#### plot truth vs prediction avg plot for each unique warmup step
#### plot truth vs prediction avg plot for each unique warmup step
for i in np.arange(len(unique_labels)):
    y_pred_mean = np.mean(y_pred[nwarmup_labels==unique_labels[i]],axis=0)
    y_truth_mean =  np.mean(y_test[nwarmup_labels==unique_labels[i]],axis=0)
    #rms_mean.append(rms(y_pred_mean-y_truth_mean))
    rms_mean.append(rms((y_pred_mean.flatten()-y_truth_mean.flatten()) / y_truth_mean.flatten()))
    print(unique_labels[i],y_pred_mean.flatten())
    y_pred_std = np.std(y_pred[nwarmup_labels==unique_labels[i]],axis=0)
    plt.errorbar(freq_sel,y_pred_mean.flatten(),yerr=y_pred_std.flatten(), ls='--', marker='o',color='blue',markersize=5,alpha=0.4,label='ML')
    #plt.title(f'Nwarmup step:{unique_labels[i]}',fontsize=20)
    #plt.plot(freq_sel,y_pred_mean,'b.-',label='prediction')
    plt.plot(freq_sel,y_truth_mean,'k-',label='DMFT')
    plt.ylabel(r' Mean Im $\Sigma_{i\omega}$ (1/eV)', fontsize=20)
    plt.xlabel('$\omega$ (eV)',fontsize=20)
    plt.legend(fontsize=15)
    plt.tight_layout()
    #plt.savefig(f'new_plots/SVO_prediction_onefourth_{unique_labels[i]}.png')
    plt.show()

    y_truth_all.append(y_truth_mean)
    y_pred_all.append(y_pred_mean)
    y_pred_std_all.append(y_pred_std)










In [ ]:
for i in range(20):
    print('nwarmup step', nwarmup_labels[i])
    plt.plot(y_pred[i],'b.-')
    plt.plot(y_test[i],'k-')
    plt.ylabel('Sigma_iw', fontsize=20)
    plt.title(f'Nwarmup step:{nwarmup_labels[i]}',fontsize=20)
    plt.tight_layout()
    #plt.savefig(f'new_plots/g02iw_{nwarmup_labels[i]}.png')
plt.show()

## 8. Save Model and RMS Summary

In [ ]:
model.save('models/trained_model_svo.keras')


In [ ]:
np.save('mean_rms_onefourth.npy',np.array([unique_labels,rms_mean]))
np.savez('onefourth_data', unique_labels=unique_labels,y_truth_all=y_truth_all,y_pred_all=y_pred_all,y_pred_std_all=y_pred_std_all)